# Capítol 2 · Treballar amb dades de text

## 2.1 Per què necessitem embeddings?

Un model no pot operar directament amb caràcters: primer convertim el text en **tokens**, després en **IDs enters** i, més endavant, en **vectors densos** (*embeddings*) que el model pot processar. Un ID només identifica un token; el valor numèric de l'ID no representa el seu significat. Els vectors s'aprendran durant l'entrenament del model.

Seguint el capítol 2 del llibre, aquest notebook compara un tokenitzador senzill construït a mà amb el BPE de GPT-2 i acaba mostrant com preparar la tasca de predir el token següent. Fem servir *The Lady with the Dog*, de Txékhov, en lloc del relat de mostra del llibre; per això canvien les paraules, els IDs i les longituds que hi veiem.

In [1]:
import re
from importlib.metadata import version
import tiktoken
from torch.utils.data import Dataset, DataLoader

from llms_from_scratch.data import read_text

## 2.2 Separar el text en tokens

`read_text` carrega el relat del directori `data/`. Inspeccionem l'inici del corpus abans de tokenitzar-lo: els salts de línia, els espais i la puntuació encara formen part del text original.

In [2]:
text = read_text("the-lady-with-the-dog.txt")
print(text[:1000])  # Mostra els primers 1.000 caràcters del text

I

It was said that a new person had appeared on the sea-front: a lady with a little dog. Dmitri Dmitritch Gurov, who had by then been a fortnight at Yalta, and so was fairly at home there, had begun to take an interest in new arrivals. Sitting in Verney’s pavilion, he saw, walking on the sea-front, a fair-haired young lady of medium height, wearing a béret; a white Pomeranian dog was running behind her.

And afterwards he met her in the public gardens and in the square several times a day. She was walking alone, always wearing the same béret, and always with the same white dog; no one knew who she was, and every one called her simply “the lady with the dog.”

“If she is here alone without a husband or friends, it wouldn’t be amiss to make her acquaintance,” Gurov reflected.

He was under forty, but he had a daughter already twelve years old, and two sons at school. He had been married young, when he was a student in his second year, and by now his wife seemed half as old again as he. 

In [3]:
result = re.split(r'([,.:;?_!"()[\]]|\s)', text)
# Eliminem espais i salts de línia; conservem les majúscules
result = [word.strip() for word in result if word.strip()]

La regex `re.split` separa en cada espai (`\s`) o signe de puntuació enumerat. Els parèntesis del patró fan que els delimitadors, com `,` i `.`, també apareguin al resultat; després `strip()` i el filtre eliminen les cadenes buides i els espais.

**No es converteix res a minúscules.** Això preserva informació com els noms propis i l'inici de frase. Aquest patró és només una demostració: no separa tots els signes possibles, com les cometes tipogràfiques, i eliminar els espais impedeix reconstruir exactament el format original.

In [4]:
print(result[:100])  # Mostra els primers 100 tokens

['I', 'It', 'was', 'said', 'that', 'a', 'new', 'person', 'had', 'appeared', 'on', 'the', 'sea-front', ':', 'a', 'lady', 'with', 'a', 'little', 'dog', '.', 'Dmitri', 'Dmitritch', 'Gurov', ',', 'who', 'had', 'by', 'then', 'been', 'a', 'fortnight', 'at', 'Yalta', ',', 'and', 'so', 'was', 'fairly', 'at', 'home', 'there', ',', 'had', 'begun', 'to', 'take', 'an', 'interest', 'in', 'new', 'arrivals', '.', 'Sitting', 'in', 'Verney’s', 'pavilion', ',', 'he', 'saw', ',', 'walking', 'on', 'the', 'sea-front', ',', 'a', 'fair-haired', 'young', 'lady', 'of', 'medium', 'height', ',', 'wearing', 'a', 'béret', ';', 'a', 'white', 'Pomeranian', 'dog', 'was', 'running', 'behind', 'her', '.', 'And', 'afterwards', 'he', 'met', 'her', 'in', 'the', 'public', 'gardens', 'and', 'in', 'the', 'square']


## 2.3 Convertir tokens en IDs

El **vocabulari** és el conjunt de tokens diferents que hem trobat al corpus. `set(result)` elimina les repeticions i `sorted(...)` fixa un ordre estable; després `enumerate` assigna un enter diferent a cada token. `all_words_size` és el nombre de tokens únics, mentre que `len(result)` comptaria també les repeticions.

Aquest vocabulari depèn del text i de les regles de separació. És **sensible a les majúscules**: `The` i `the` poden tenir IDs diferents. Els IDs són etiquetes arbitràries; encara no són els vectors que rebrà la xarxa neuronal.

In [5]:
all_words = sorted(set(result))
all_words_size = len(all_words)

In [6]:
vocabulary = {word: i for i, word in enumerate(all_words)}
print(vocabulary)

{'!': 0, '(': 1, ')': 2, ',': 3, '.': 4, ':': 5, ';': 6, '?': 7, 'A': 8, 'After': 9, 'Afterwards': 10, 'All': 11, 'Amphitheatre': 12, 'And': 13, 'Anna': 14, 'Arts': 15, 'As': 16, 'At': 17, 'Bazaar': 18, 'Belyov': 19, 'Both': 20, 'But': 21, 'Complete': 22, 'Council': 23, 'Crown': 24, 'December': 25, 'Department': 26, 'Diderits': 27, 'Dimitri': 28, 'Dmitri': 29, 'Dmitritch': 30, 'Don’t': 31, 'During': 32, 'Evil': 33, 'Experience': 34, 'From': 35, 'Geisha”': 36, 'German': 37, 'Go': 38, 'God': 39, 'Gontcharny': 40, 'Governor': 41, 'Governor’s': 42, 'Grenada': 43, 'Gurov': 44, 'Had': 45, 'He': 46, 'Her': 47, 'Here': 48, 'His': 49, 'How': 50, 'I': 51, 'II': 52, 'III': 53, 'IV': 54, 'If': 55, 'In': 56, 'It': 57, 'It’s': 58, 'I’ll': 59, 'I’ve': 60, 'Japanese': 61, 'Just': 62, 'Left': 63, 'Let': 64, 'Little': 65, 'Moscow': 66, 'Most': 67, 'My': 68, 'No': 69, 'Oh': 70, 'Old': 71, 'On': 72, 'Once': 73, 'One': 74, 'Oreanda': 75, 'Orthodox': 76, 'Owing': 77, 'Petersburg': 78, 'Petrovka': 79, 'Pomer

In [7]:
class SimpleTokenizer:
    def __init__(self, vocabulary):
        self.str_to_idx = vocabulary
        self.idx_to_str = {i: word for word, i in vocabulary.items()}
    def encode(self, text):
        tokens = re.split(r'([,.:;?_!"()[\]]|\s)', text)
        tokens = [word.strip() for word in tokens if word.strip()]
        return [self.str_to_idx[word] for word in tokens if word in self.str_to_idx]

    def decode(self, token_ids):
        text = ' '.join(self.idx_to_str[token_id] for token_id in token_ids if token_id in self.idx_to_str)
        text = re.sub(r'\s([,.:;?_!"()[\]])', r'\1', text)  # Elimina l’espai abans de la puntuació
        return text

`SimpleTokenizer.encode` rep text, el separa amb les mateixes regles i consulta `str_to_idx` per obtenir un ID per token. `decode` fa el camí invers amb `idx_to_str`; la substitució final treu espais artificials davant de la puntuació.

La descodificació recupera una versió llegible, però no necessàriament idèntica a l'entrada: s'han perdut els espais i els salts de línia originals. A més, el `if word in self.str_to_idx` d'`encode` **descarta els tokens desconeguts** sense avisar; l'exemple següent ho farà visible.

In [8]:
tokenizer = SimpleTokenizer(vocabulary)
example = "The lady with the dog."
encoded = tokenizer.encode(example)
print("Encoded:", encoded)
decoded = tokenizer.decode(encoded)
print("Decoded:", decoded)

Encoded: [99, 824, 1537, 1384, 456, 4]
Decoded: The lady with the dog.


In [9]:
tokenizer = SimpleTokenizer(vocabulary)
example = "The lady with the dog dogaaaaaaa."
encoded = tokenizer.encode(example)
print("Encoded:", encoded)
decoded = tokenizer.decode(encoded)
print("Decoded:", decoded)

Encoded: [99, 824, 1537, 1384, 456, 4]
Decoded: The lady with the dog.


En el primer exemple, totes les peces són al vocabulari i la frase es reconstrueix. En el segon, `dogaaaaaaa` no hi és: el `SimpleTokenizer` l'elimina i `decode` retorna una frase sense aquesta paraula. Que una frase sembli llegible no garanteix que s'hagi conservat tota la informació; cal comprovar explícitament els tokens desconeguts.

## 2.4 Afegir tokens especials

Per no perdre la posició dels tokens desconeguts, ampliem el vocabulari amb `<|unk|>`. També hi afegim `<|endoftext|>` per marcar el límit entre fragments de text independents. Com que els afegim al final, els IDs dels tokens anteriors es mantenen.

Aquests marcadors són convencions del tokenitzador i del conjunt de dades. El model només en veu els IDs i n'aprèn l'ús a partir dels exemples d'entrenament.

In [10]:
all_tokens = sorted(list(set(result)))
all_tokens.extend(['<|endoftext|>', '<|unk|>'])
vocabulary = {token: i for i, token in enumerate(all_tokens)}
print(vocabulary)

{'!': 0, '(': 1, ')': 2, ',': 3, '.': 4, ':': 5, ';': 6, '?': 7, 'A': 8, 'After': 9, 'Afterwards': 10, 'All': 11, 'Amphitheatre': 12, 'And': 13, 'Anna': 14, 'Arts': 15, 'As': 16, 'At': 17, 'Bazaar': 18, 'Belyov': 19, 'Both': 20, 'But': 21, 'Complete': 22, 'Council': 23, 'Crown': 24, 'December': 25, 'Department': 26, 'Diderits': 27, 'Dimitri': 28, 'Dmitri': 29, 'Dmitritch': 30, 'Don’t': 31, 'During': 32, 'Evil': 33, 'Experience': 34, 'From': 35, 'Geisha”': 36, 'German': 37, 'Go': 38, 'God': 39, 'Gontcharny': 40, 'Governor': 41, 'Governor’s': 42, 'Grenada': 43, 'Gurov': 44, 'Had': 45, 'He': 46, 'Her': 47, 'Here': 48, 'His': 49, 'How': 50, 'I': 51, 'II': 52, 'III': 53, 'IV': 54, 'If': 55, 'In': 56, 'It': 57, 'It’s': 58, 'I’ll': 59, 'I’ve': 60, 'Japanese': 61, 'Just': 62, 'Left': 63, 'Let': 64, 'Little': 65, 'Moscow': 66, 'Most': 67, 'My': 68, 'No': 69, 'Oh': 70, 'Old': 71, 'On': 72, 'Once': 73, 'One': 74, 'Oreanda': 75, 'Orthodox': 76, 'Owing': 77, 'Petersburg': 78, 'Petrovka': 79, 'Pomer

In [11]:
class ImprovedSimpleTokenizer:
    def __init__(self, vocabulary):
        self.str_to_idx = vocabulary
        self.idx_to_str = {i: word for word, i in vocabulary.items()}
    def encode(self, text):
        tokens = re.split(r'([,.:;?_!"()[\]]|\s)', text)
        tokens = [word.strip() for word in tokens if word.strip()]
        return [self.str_to_idx.get(word, self.str_to_idx['<|unk|>']) for word in tokens]
    def decode(self, token_ids):
        text = ' '.join(self.idx_to_str.get(token_id, '<|unk|>') for token_id in token_ids)
        text = re.sub(r'\s([,.:;?_!"()[\]])', r'\1', text)  # Elimina l’espai abans de la puntuació
        return text

`ImprovedSimpleTokenizer` conserva tots els tokens de l'entrada: `dict.get(word, id_unk)` substitueix els desconeguts per l'ID de `<|unk|>`. `decode` pot recuperar on hi havia un desconegut, encara que **no en pot recuperar la grafia original**.

El token `<|endoftext|>` ja té una entrada pròpia al vocabulari i passa intacte per aquesta regex; el model el podria fer servir per reconèixer el pas d'un document a un altre.

In [12]:
text1 = text[:89] + " TEST"
text2 = text[89:200]
joined = " <|endoftext|> ".join([text1, text2])
print(joined)

I

It was said that a new person had appeared on the sea-front: a lady with a little dog. TEST <|endoftext|>  Dmitri Dmitritch Gurov, who had by then been a fortnight at Yalta, and so was fairly at home there, had begun 


In [13]:
tokenizer = ImprovedSimpleTokenizer(vocabulary)
encoded = tokenizer.encode(joined)
print("Encoded:", encoded)
decoded = tokenizer.decode(encoded)
print("Decoded:", decoded)

Encoded: [51, 57, 1498, 1169, 1381, 122, 965, 1059, 676, 175, 997, 1384, 1188, 5, 122, 824, 1537, 122, 861, 456, 4, 1613, 1612, 29, 30, 44, 3, 1525, 676, 284, 1391, 220, 122, 608, 190, 119, 3, 163, 1269, 1498, 545, 190, 725, 1392, 3, 676, 227]
Decoded: I It was said that a new person had appeared on the sea-front: a lady with a little dog. <|unk|> <|endoftext|> Dmitri Dmitritch Gurov, who had by then been a fortnight at Yalta, and so was fairly at home there, had begun


`joined` uneix dos fragments amb `<|endoftext|>` per simular un límit de document. `TEST`, afegit expressament al primer fragment, no apareix al corpus i es codifica com `<|unk|>`. La descodificació mostra tots dos marcadors, però normalitza els espais i els salts de línia. Aquesta és la diferència respecte del primer tokenitzador, que feia desaparèixer la paraula desconeguda.

## 2.5 Byte pair encoding (BPE)

El tokenitzador manual depèn d'un vocabulari de paraules completes. El **BPE de GPT-2**, disponible amb `tiktoken`, combina seqüències de bytes freqüents en tokens; una paraula nova es pot representar amb diversos tokens més petits. Així no necessita un `<|unk|>` per a paraules desconegudes.

`tiktoken.get_encoding("gpt2")` carrega les regles i el vocabulari de GPT-2. No hem entrenat aquest tokenitzador amb el nostre relat: els seus IDs són diferents dels del diccionari `vocabulary` que acabem de construir. La versió de `tiktoken` s'imprimeix per facilitar la reproducció de l'exemple.

In [14]:
print(tiktoken.__version__)

0.14.0


In [15]:
tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode(joined, allowed_special={"<|endoftext|>"})
print("Encoded:", encoded)
decoded = tokenizer.decode(encoded)
print("Decoded:", decoded)

Encoded: [40, 198, 198, 1026, 373, 531, 326, 257, 649, 1048, 550, 4120, 319, 262, 5417, 12, 8534, 25, 257, 10846, 351, 257, 1310, 3290, 13, 43001, 220, 50256, 220, 35438, 380, 35438, 46510, 402, 1434, 85, 11, 508, 550, 416, 788, 587, 257, 46327, 379, 575, 2501, 64, 11, 290, 523, 373, 6547, 379, 1363, 612, 11, 550, 9258, 220]
Decoded: I

It was said that a new person had appeared on the sea-front: a lady with a little dog. TEST <|endoftext|>  Dmitri Dmitritch Gurov, who had by then been a fortnight at Yalta, and so was fairly at home there, had begun 


Aquí `tokenizer` ja és el BPE de GPT-2. `allowed_special={"<|endoftext|>"}` autoritza que aquesta cadena es tracti com el token especial de separació; sense autoritzar-la, `tiktoken` rebutjaria la crida en aquest cas. A GPT-2, el seu ID és `50256`.

El BPE conserva el text amb molta més fidelitat, inclosos els salts de línia. També pot codificar `TEST` sense `<|unk|>`, dividint-lo si cal en tokens més petits. Que els IDs i el nombre de tokens siguin diferents dels del tokenitzador manual és normal: cada tokenitzador defineix les seves pròpies unitats.

## 2.6 Mostreig de dades amb una finestra lliscant

In [16]:
encoded_text = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(f"Number of tokens in the text: {len(encoded_text)}\nLength of the text: {len(text)}")

Number of tokens in the text: 9016
Length of the text: 36361


In [17]:
encoded_sample = encoded_text[787:]

`encoded_text` és el relat complet convertit a IDs amb el BPE. El nombre de tokens no coincideix amb el de caràcters: un token pot representar una paraula, una part de paraula, un espai o una altra seqüència de bytes. Les xifres d'aquest corpus no han de coincidir amb les del relat utilitzat al llibre.

`encoded_text[787:]` salta els primers 787 tokens per triar un passatge il·lustratiu. Aquest desplaçament no és un hiperparàmetre del model; només determina quin fragment veiem en els exemples següents.

In [18]:
cs = 5 # mida màxima del context en aquest exemple
for i in range(1, cs+1):
    context = encoded_sample[:i]
    desired = encoded_sample[i]
    print(f"Context: {context}, Desired: {desired}")

Context: [484], Desired: 550
Context: [484, 550], Desired: 587
Context: [484, 550, 587], Desired: 1498
Context: [484, 550, 587, 1498], Desired: 26
Context: [484, 550, 587, 1498, 26], Desired: 475


In [19]:
for i in range(1, cs+1):
    context = encoded_sample[:i]
    desired = encoded_sample[i]
    print(f"Context: {tokenizer.decode(context)}, Desired: {tokenizer.decode([desired])}")

Context:  they, Desired:  had
Context:  they had, Desired:  been
Context:  they had been, Desired:  able
Context:  they had been able, Desired: ;
Context:  they had been able;, Desired:  but


### Context i objectiu de predicció

En cada iteració, `context = encoded_sample[:i]` conté els primers `i` tokens i `desired = encoded_sample[i]` és **el token immediatament posterior**. Amb `cs = 5` mostrem cinc prediccions: el context creix d'un a cinc tokens. `decode([desired])` rep una llista perquè `decode` espera una seqüència d'IDs.

Aquest bucle només **il·lustra** la tasca de predir el token següent. Per entrenar amb finestres de longitud fixa, una entrada seria $x=(t_0,\ldots,t_{c-1})$ i els objectius serien els mateixos tokens desplaçats una posició, $y=(t_1,\ldots,t_c)$. La finestra es podria moure pel corpus amb un pas determinat i agrupar els parells en batches.